# Promoter pausing and accessibility analysis 

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
import pysam
import pyBigWig
from collections import defaultdict
import os
import os

the tss accessibility table is located at `/project/spott/cshan/fiber-seq/results/PolII/m6a_pausing_quartiles/AL10_bc2178_19130_1kb_bin10_modthresh0.9_all/tables/AL10_bc2178_19130_tss_read_accessibility.tsv.gz`
- the code used to create this table is located at `/project/spott/cshan/fiber-seq/code/PolII_footprints_code/m6a_pausing_quartiles`

In [1]:
%%bash

less /project/spott/cshan/fiber-seq/results/PolII/m6a_pausing_quartiles/AL10_bc2178_19130_1kb_bin10_modthresh0.9_all/tables/AL10_bc2178_19130_tss_read_accessibility.tsv.gz | head

read_id	tss_uid	gene_id	gene_id_base	gene_name	chromosome	tss_coordinate	tss_strand	tss_source	PI	pausing_group	n_positions_covered	n_m6a_modified	n_m6a_unmodified	tss_accessible
AL10_bc2178_19130-m84241_250311_201728_s2/64488132/ccs	chr1|6206118|+|ENSG00000158286|185	ENSG00000158286.14	ENSG00000158286	RNF207	chr1	6206118	+	CAGE	7.99178294573734	Q4 (High PI)	60	15	23	TRUE
AL10_bc2178_19130-m84241_250311_201728_s2/125570575/ccs	chr1|6206118|+|ENSG00000158286|185	ENSG00000158286.14	ENSG00000158286	RNF207	chr1	6206118	+	CAGE	7.99178294573734	Q4 (High PI)	60	8	14	TRUE
AL10_bc2178_19130-m84241_250311_201728_s2/218368972/ccs	chr1|6206118|+|ENSG00000158286|185	ENSG00000158286.14	ENSG00000158286	RNF207	chr1	6206118	+	CAGE	7.99178294573734	Q4 (High PI)	60	8	14	TRUE
AL10_bc2178_19130-m84241_250311_201728_s2/219547243/ccs	chr1|6206118|+|ENSG00000158286|185	ENSG00000158286.14	ENSG00000158286	RNF207	chr1	6206118	+	CAGE	7.99178294573734	Q4 (High PI)	60	0	22	FALSE
AL10_bc2178_19130-m84241_250311_2017

This table contains one row per read-TSS pair, summarizing the m6A modification status of adenines within ±100 bp of the TSS for each read. It includes the following columns:

| Column | Description | Example Value | Type |
|--------|-------------|----------------|------|
| `read_id` | Unique identifier for the sequenced read (PacBio CCS) | `AL10_bc2178_19130-m84241_250311_201728_s2/64488132/ccs` | string |
| `tss_uid` | Unique TSS identifier built as chr\|tss_coord\|strand\|gene_id\|row_id | `chr1\|6206118\|+\|ENSG00000158286\|185` | string |
| `gene_id` | GENCODE gene identifier with version suffix | `ENSG00000158286.14` | string |
| `gene_id_base` | GENCODE gene identifier without version suffix | `ENSG00000158286` | string |
| `gene_name` | HGNC gene symbol | `RNF207` | string |
| `chromosome` | Genomic chromosome | `chr1` | string |
| `tss_coordinate` | Transcription start site position (1-based) | `6206118` | integer |
| `tss_strand` | Strand orientation of the gene | `+` | string (+/-) |
| `tss_source` | TSS source: CAGE or GENCODE | `CAGE` | string |
| `PI` | Pol II pausing index (higher = more paused) | `7.99178294573734` | float |
| `pausing_group` | Quartile assignment based on PI | `Q4 (High PI)` | string (Q1-Q4) |
| `n_positions_covered` | Number of adenine positions covered by this read within ±100 bp of TSS | `60` | integer |
| `n_m6a_modified` | Number of m6A-modified adenines detected in this read within ±100 bp of TSS | `15` | integer |
| `n_m6a_unmodified` | Number of unmodified adenines detected in this read within ±100 bp of TSS | `23` | integer |
| `tss_accessible` | Boolean: TRUE if ≥1 m6A call in ±100 bp window (chromatin open), FALSE if 0 m6A calls (protected/inaccessible) | `TRUE` | boolean |

**Key points:**
- Each row = one read's experience at one TSS
- `n_positions_covered` = `n_m6a_modified` + `n_m6a_unmodified` (all called adenines in the ±100 bp window)
- `tss_accessible = TRUE` when a read carries m6A evidence near the TSS; `FALSE` means the region was inaccessible or protected from methylation


# table for footprint midpoints
example table at: `/project/spott/cshan/fiber-seq/results/PolII/footprint_summary_beds/FIRE_nucleosome/chr1_footprints.bed.gz`
- the code used to create this table is located at `/project/spott/cshan/fiber-seq/code/PolII_footprints_code/footprints_summary_code`

In [2]:
%%bash

less /project/spott/cshan/fiber-seq/results/PolII/footprint_summary_beds/FIRE_nucleosome/chr1_footprints.bed.gz | head -n 10

chr1	10000	10018	m84241_250909_000942_s2/264508392/ccs	FIRE_nucleosome	10009.0
chr1	10000	10020	m84241_250908_220650_s1/250481348/ccs	FIRE_nucleosome	10010.0
chr1	10000	10024	m84241_250326_031141_s3/130221495/ccs	FIRE_nucleosome	10012.0
chr1	10000	10030	m84241_250514_222554_s2/266278306/ccs	FIRE_nucleosome	10015.0
chr1	10000	10044	m84241_250326_010847_s2/18220681/ccs	FIRE_nucleosome	10022.0
chr1	10000	10048	m84241_250906_015218_s2/106303285/ccs	FIRE_nucleosome	10024.0
chr1	10000	10050	m84241_250908_200356_s4/54790654/ccs	FIRE_nucleosome	10025.0
chr1	10000	10056	m84241_250326_010847_s2/156568420/ccs	FIRE_nucleosome	10028.0
chr1	10000	10056	m84241_250906_015218_s2/215484369/ccs	FIRE_nucleosome	10028.0
chr1	10000	10062	m84241_250905_234909_s1/164431819/ccs	FIRE_nucleosome	10031.0


# For reads that have accessibility at TSS (+/- 100 bp), how many nucleosome/PPP footprints do they have within 1 kb of the TSS?

1. First, I want to merge the footprint summary files for nucleosomes called by FIRE. The files are separated by chromosomes at `/project/spott/cshan/fiber-seq/results/PolII/footprint_summaries/FIRE_nucleosome`

In [2]:
%%bash

cat << 'EOF' > /project/spott/cshan/fiber-seq/code/PolII_footprints_code/accessibility_pausing_code/merge_FIRE_nucleosome.sh
#!/bin/bash

#SBATCH --job-name=merge_FIRE_nuc
#SBATCH --account=pi-spott
#SBATCH --partition=bigmem
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=50G
#SBATCH --time=12:00:00
#SBATCH --output=/project/spott/cshan/fiber-seq/results/logs/merge_FIRE_nuc_%j.out
#SBATCH --error=/project/spott/cshan/fiber-seq/results/logs/merge_FIRE_nuc_%j.err


INPUT_DIR="/project/spott/cshan/fiber-seq/results/PolII/footprint_summary_beds/FIRE_nucleosome"
OUTPUT_FILE="${INPUT_DIR}/merged_footprint_summary.tsv.gz"

echo "Starting merge at $(date)"

cd "$INPUT_DIR"

# create header
echo -e "chrom\tstart\tend\tread_id\tclass\tmidpoint" | gzip > "$OUTPUT_FILE"

# append all files (skip headers since BED doesn't have one)
for f in *.bed.gz; do
    echo "Processing $f"
    zcat "$f" >> temp_merged.tsv
done

# compress final
gzip -c temp_merged.tsv >> "$OUTPUT_FILE"

# cleanup
rm temp_merged.tsv

echo "Finished at $(date)"
EOF

In [3]:
!chmod +x /project/spott/cshan/fiber-seq/code/PolII_footprints_code/accessibility_pausing_code/merge_FIRE_nucleosome.sh

In [4]:
!sbatch /project/spott/cshan/fiber-seq/code/PolII_footprints_code/accessibility_pausing_code/merge_FIRE_nucleosome.sh

sbatch: Verify job submission ...
sbatch: Using a shared partition ...
sbatch: Partition: bigmem
sbatch: QOS-Flag: bigmem
sbatch: Account: pi-spott
sbatch: Verification: ***PASSED***
Submitted batch job 49271472


### Selected genes for high/low pausing (nucleosome midpoint)

Genes were selected based on their ranked positions from:
`/project/spott/cshan/fiber-seq/code/PolII_footprints_code/plot_polii_footprints_precomputed_sort.py`

| High pausing index | Low pausing index |
|--------------------|-------------------|
| ZCWPW1            | ENO1              |
| ARFGEF2           | ZNF75A            |
| ICE2              | KCNC3             |
| B2M               | NECTIN1           |
| MTF2              | SLC2A1            |